# MB52 | Refinamento da Fase 4 | Centro 4128

Notebook para exibir a saída completa de `MB52_Fase4_Comparacao_4128.xlsx`: inventário, todas as abas, preenchimento, cobertura, duplicidades, distribuições, alertas e exportação consolidada.

In [0]:
%sql
from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)
ARQUIVO=Path('MB52_Fase4_Comparacao_4128.xlsx')
PASTA_SAIDA=Path('saida_refinamento_4128'); PASTA_SAIDA.mkdir(exist_ok=True)
if not ARQUIVO.exists():
    candidatos=list(Path('.').glob('*MB52*4128*.xlsx'))
    if candidatos: ARQUIVO=candidatos[0]
    else: raise FileNotFoundError('Coloque MB52_Fase4_Comparacao_4128.xlsx na mesma pasta deste notebook.')
print('Arquivo:', ARQUIVO.resolve())

## 1. Leitura, inventário e prévia de todas as abas

In [0]:
abas=pd.read_excel(ARQUIVO,sheet_name=None,engine='openpyxl')
inventario=pd.DataFrame([{'aba':n,'linhas':len(d),'colunas':len(d.columns),'linhas_vazias':int(d.isna().all(axis=1).sum()),'colunas_vazias':int(d.isna().all(axis=0).sum()),'duplicatas_exatas':int(d.duplicated().sum())} for n,d in abas.items()])
display(inventario)
for n,d in abas.items():
    display(Markdown(f'### `{n}` | {len(d):,} linhas x {len(d.columns):,} colunas'))
    display(d.head(20))

## 2. Catálogo completo de colunas e preenchimento

In [0]:
def uteis(s):
    x=s.astype('string').str.strip().str.upper()
    return int((~(x.isna()|x.isin(['','NAN','NONE','NULL','<NA>']))).sum())
cat=[]
for aba,df in abas.items():
    for c in df.columns:
        u=uteis(df[c]); cat.append({'aba':aba,'coluna':str(c),'tipo':str(df[c].dtype),'linhas':len(df),'valores_uteis':u,'pct_preenchido':u/len(df) if len(df) else np.nan,'distintos':int(df[c].nunique(dropna=False)),'exemplos':' | '.join(df[c].dropna().astype(str).drop_duplicates().head(5))})
catalogo=pd.DataFrame(cat).sort_values(['aba','pct_preenchido','coluna'])
display(catalogo.style.format({'pct_preenchido':'{:.1%}'}))
display(Markdown('### Colunas 100% vazias'))
display(catalogo[catalogo.valores_uteis.eq(0)])

## 3. Cobertura de colunas entre todas as abas

In [0]:
def norm(x):
    x=unicodedata.normalize('NFKD',str(x)).encode('ascii','ignore').decode().upper()
    return re.sub(r'[^A-Z0-9]+','',x)
mapas={a:{norm(c):c for c in d.columns} for a,d in abas.items()}
comp=[]; nomes=list(abas)
for i,a in enumerate(nomes):
    for b in nomes[i+1:]:
        ca,cb=set(mapas[a]),set(mapas[b]); comp.append({'aba_a':a,'aba_b':b,'comuns':len(ca&cb),'somente_a':len(ca-cb),'somente_b':len(cb-ca),'lista_somente_a':', '.join(sorted(ca-cb)),'lista_somente_b':', '.join(sorted(cb-ca))})
cobertura=pd.DataFrame(comp).sort_values('comuns',ascending=False) if comp else pd.DataFrame()
display(cobertura)

## 4. Duplicidades e chaves candidatas

In [0]:
PADROES=['MATERIAL','MATNR','CENTRO','WERKS','DEPOSITO','LGORT','LOTE','CHARG','TIPOAVALIACAO','BWTAR']
def chaves(df): return [c for c in df.columns if any(p in norm(c) for p in PADROES)]
rd=[]
for a,d in abas.items():
    k=chaves(d); rd.append({'aba':a,'chaves':', '.join(map(str,k)),'duplicatas_exatas':int(d.duplicated(keep=False).sum()),'repetidas_na_chave':int(d.duplicated(k,keep=False).sum()) if k else np.nan})
    display(Markdown(f'### `{a}` | chave candidata: {k or "não identificada"}'))
    if k: display(d[d.duplicated(k,keep=False)].sort_values(k).head(200))
    if d.duplicated().any(): display(d[d.duplicated(keep=False)].head(200))
resumo_duplicidades=pd.DataFrame(rd); display(resumo_duplicidades)

## 5. Distribuições e extremos

In [0]:
for a,d in abas.items():
    display(Markdown(f'### `{a}`'))
    num=d.select_dtypes(include='number')
    if not num.empty: display(num.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)
    for c in d.columns:
        if d[c].nunique(dropna=False)<=30:
            display(Markdown(f'**Frequência: `{c}`**')); display(d[c].value_counts(dropna=False).head(30).to_frame('quantidade'))

## 6. Saída completa das abas de resultado

In [0]:
termos=('DIVERG','ERRO','AUSENT','EXTRA','RESUM','COMPAR','AUDIT','MATCH','DUPLIC')
resultados={n:d for n,d in abas.items() if any(t in norm(n) for t in termos)} or abas
for n,d in resultados.items():
    display(Markdown(f'### Saída completa: `{n}` | {len(d):,} registros'))
    display(d)
    d.to_csv(PASTA_SAIDA/f'{norm(n)[:80]}.csv',index=False,sep=';',encoding='utf-8-sig')

## 7. Alertas e exportação consolidada

In [0]:
alertas=[]
for a,d in abas.items():
    vaz=[str(c) for c in d.columns if uteis(d[c])==0]
    if vaz: alertas.append({'aba':a,'tipo':'COLUNA_100_PCT_VAZIA','quantidade':len(vaz),'detalhe':', '.join(vaz)})
    if d.duplicated().any(): alertas.append({'aba':a,'tipo':'LINHAS_IDENTICAS','quantidade':int(d.duplicated(keep=False).sum()),'detalhe':'Revisar duplicidade real de carga.'})
    k=chaves(d)
    if k and d.duplicated(k).any(): alertas.append({'aba':a,'tipo':'CHAVE_REPETIDA','quantidade':int(d.duplicated(k,keep=False).sum()),'detalhe':'Comparar campo a campo antes de classificar como duplicidade.'})
alertas_df=pd.DataFrame(alertas); display(alertas_df if not alertas_df.empty else Markdown('Nenhum alerta automático.'))
with pd.ExcelWriter(PASTA_SAIDA/'MB52_4128_Saida_Refinamento.xlsx',engine='openpyxl') as w:
    inventario.to_excel(w,sheet_name='Inventario',index=False); catalogo.to_excel(w,sheet_name='Catalogo_Colunas',index=False); cobertura.to_excel(w,sheet_name='Cobertura_Abas',index=False); resumo_duplicidades.to_excel(w,sheet_name='Duplicidades',index=False); alertas_df.to_excel(w,sheet_name='Alertas',index=False)
print('Saídas:', PASTA_SAIDA.resolve())

## Leitura crítica

- Coluna vazia no Datalake só é perda se houver valor útil no SAP.
- Chave repetida exige comparação campo a campo para separar duplicidade real de granularidade adicional.
- Volumetria igual não comprova qualidade.
- Resultado com 0% ou 100% de match exige revisão de chave, parsing, coluna vazia e normalização.